## Week 2 — Deep research (Samuel Adebodun)

Clarify → plan 3 searches (`WebSearchTool`) → structured report. [Web search pricing](https://platform.openai.com/docs/pricing#web-search).

Kernel cwd: `SamuelAdebodun`. `.env`: next cell finds repo root (`agents/` with `2_openai/`).

In [ ]:
%pip install -q openai-agents python-dotenv pydantic gradio

In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv

REPO_ROOT = None
for d in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (d / "2_openai").is_dir() and (d / ".env").is_file():
        REPO_ROOT = d
        break
if REPO_ROOT is not None:
    load_dotenv(REPO_ROOT / ".env", override=True, encoding="utf-8-sig")
else:
    load_dotenv(override=True, encoding="utf-8-sig")

if not Path("deep_research.py").exists():
    raise RuntimeError(
        "Set the notebook working directory to SamuelAdebodun (where deep_research.py lives)."
    )

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY missing. Put it in agents/.env next to 2_openai/ (course default). "
        f"REPO_ROOT found: {REPO_ROOT}. Keys: https://platform.openai.com/api-keys"
    )

In [ ]:
from IPython.display import Markdown, display

from deep_research import ResearchManager

### 1) Web search

Uses OpenAI **`WebSearchTool`** (hosted), same idea as `4_lab4.ipynb`.

In [ ]:
RESEARCH_QUERY = (
    "What should a DevOps/platform team prioritize when hardening AKS clusters in 2026?"
)

### 2) Clarifying questions

Scope the topic before searching.

In [ ]:
mgr = ResearchManager()
clarified = await mgr.clarify(RESEARCH_QUERY)

for i, q in enumerate(clarified.questions, start=1):
    print(f"{i}. {q}")

### 3) Your answers

Three strings — guides the search plan.

In [ ]:
MY_ANSWERS = [
    "Mid-size company on Azure, regulated-ish (need audit-friendly controls).",
    "2024–2025 guidance and AKS-specific features.",
    "Practical checklist level, not academic survey.",
]

### 4) Run pipeline

Last stream chunk is the report; earlier lines are status.

In [ ]:
chunks: list[str] = []
async for part in mgr.run(RESEARCH_QUERY, clarified.questions, MY_ANSWERS):
    chunks.append(part)
    if not part.startswith("Trace:") and "trace?trace_id=" not in part:
        print(part[:200] + ("…" if len(part) > 200 else ""))

display(Markdown(chunks[-1]))

### 5) Optional `deep_research` imports

Uncomment lines in the next cell for direct access to agents or schemas (instead of only `ResearchManager`).

In [ ]:
# from deep_research import ResearchManager
# from deep_research import clarifier_agent, planner_agent, search_agent, writer_agent
# from deep_research import ClarifyingQuestions, WebSearchPlan, WebSearchItem, ReportData
